# pulmonary_colab_training.ipynb

**Repository:** pulmonary_nodule_verification v1.0.0 · **Date:** 2026-10-26 · **Developer:** Prof. Wesley Pacheco Calixto · **License:** MIT · **Corresponding author:** Wesley Pacheco Calixto, wesley.compute-node

**Citation:** Calixto, W. P., Camargo, T. F. O., Pacheco, V. M. G., Martins, M. S. R., & Coimbra, A. P. (2026). Multiscale morphology-informed post-detection verification of pulmonary nodule candidates in CT: code, trained nnU-Net models, cohort splits and analysis data (v1.0.0) [Data set and software]. Zenodo. https://doi.org/10.5281/zenodo.23674673

Outputs have been cleared; machine-specific paths were neutralised. See docs/file_level_documentation.md.


# Pulmonary — Treino nnU-Net no Google Colab (H100)

## Como usar
1. `Runtime` → `Change runtime type` → **GPU** → **H100**
2. Executar **A → B → C → D** (sempre ao iniciar sessão nova)
3. Executar **TREINAR TUDO** — corre toda a pipeline, saltando o que já está feito
4. Se a sessão for interrompida: re-executar A → D e voltar a correr **TREINAR TUDO**

## Arquitecturas e ordem de treino
| Fase | Arquitectura | Trainer | Pré-requisito |
|---|---|---|---|
| 1 | 3d_lowres nnUNetPlans | Foreground100 | — |
| 2 | 3d_lowres nnUNetResEncUNetMPlans | Foreground100 | — |
| 3 | 2D nnUNetPlans | padrão | — |
| 4 | 2D nnUNetResEncUNetMPlans (2D_REM) | padrão | — |
| 5 | 2D nnUNetResEncUNetLPlans (2D_REL) | padrão | — |
| 6 | 3d_fullres nnUNetPlans (3D_F) | padrão | — |
| 7 | 3d_fullres nnUNetResEncUNetMPlans (3D_FREM) | padrão | — |
| 8 | cascade nnUNetPlans (3D_CF) | Foreground100 | Fase 1 + 6 completos |
| 9 | cascade nnUNetResEncUNetMPlans (3D_CFREM) | Foreground100 | Fase 2 + 7 completos |

## Sobre os folds do RunPod
- `nnUNetTrainer__nnUNetPlans__3d_lowres` folds 0, 1: **DEGENERADOS** (Pseudo dice = 0) — ficam **intocados** como evidência científica
- `nnUNetTrainer_Foreground100__nnUNetPlans__3d_lowres`: **pasta nova**, nenhum fold existe ainda — todos os 5 folds serão treinados do zero
- `nnUNetTrainer_Foreground100__nnUNetResEncUNetMPlans__3d_lowres` fold 1: RunPod parou a ~epoch 420 → **retomado** automaticamente a partir do `checkpoint_latest`
- fold 0 do mesmo: **falhou** no RunPod → começa do zero

---
## Célula A — GPU, Drive e nnU-Net
**Execute sempre ao iniciar sessão.**

In [ ]:
import subprocess, sys

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', gpu.stdout.strip())

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

try:
    import nnunetv2
    try:
        import importlib.metadata
        version = importlib.metadata.version('nnunetv2')
    except Exception:
        version = getattr(nnunetv2, '__version__', 'instalado')
    print(f'nnU-Net já instalado: {version}')
except ImportError:
    print('Instalando nnU-Net v2.8.0...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'nnunetv2==2.8.0', '-q'], check=True)
    print('nnU-Net instalado.')

print('Célula A OK')

---
## Célula B — Trainer Foreground100
**Execute sempre ao iniciar sessão** (não persiste entre sessões).

Cria o trainer com `oversample_foreground_percent = 1.0` para resolver a degeneração:
135 casos sem anotação (16% do dataset) + nódulos < 1% dos voxels → o trainer padrão
(33% oversampling) converge para prever tudo como fundo.

In [ ]:
import os, sys, subprocess, nnunetv2

variants_dir = os.path.join(
    os.path.dirname(nnunetv2.__file__),
    'training', 'nnUNetTrainer', 'variants'
)

trainer_code = '''import torch
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer


class nnUNetTrainer_Foreground100(nnUNetTrainer):
    """100% foreground oversampling para nodulos pulmonares pequenos."""

    def __init__(self, plans: dict, configuration: str, fold: int,
                 dataset_json: dict, device: torch.device = torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.oversample_foreground_percent = 1.0
'''

with open(os.path.join(variants_dir, 'nnUNetTrainer_Foreground100.py'), 'w') as f:
    f.write(trainer_code)

result = subprocess.run(
    [sys.executable, '-c',
     'from nnunetv2.training.nnUNetTrainer.variants'
     '.nnUNetTrainer_Foreground100 import nnUNetTrainer_Foreground100; print("OK")'],
    capture_output=True, text=True
)
print('nnUNetTrainer_Foreground100:', result.stdout.strip() or result.stderr.strip())

---
## Célula C — Caminhos e funções auxiliares
**Execute sempre ao iniciar sessão.**

In [ ]:
import os, sys, shutil, subprocess, threading, time

DRIVE     = '/content/drive/MyDrive/Pulmonary'
DRIVE_PRE = os.path.join(DRIVE, 'nnUNet_preprocessed')
DRIVE_RES = os.path.join(DRIVE, 'nnUNet_results')

LOCAL_PRE = '/content/nnUNet_preprocessed'
LOCAL_RAW = '/content/nnUNet_raw'
LOCAL_RES = '/content/nnUNet_results'

DS_ID   = 1
DS_NAME = 'Dataset001_PulmonaryNodules'
DS_PRE  = os.path.join(LOCAL_PRE, DS_NAME)
DS_RES  = os.path.join(LOCAL_RES, DS_NAME)

for d in [DS_PRE, LOCAL_RAW, DS_RES]:
    os.makedirs(d, exist_ok=True)

os.environ['nnUNet_raw']          = LOCAL_RAW
os.environ['nnUNet_preprocessed'] = LOCAL_PRE
os.environ['nnUNet_results']      = LOCAL_RES


def sync_to_drive(silent=False):
    """Copia /content/nnUNet_results → Drive."""
    if not silent:
        print('    [sync] copiando para Drive...', end=' ', flush=True)
    os.makedirs(DRIVE_RES, exist_ok=True)
    subprocess.run(['cp', '-ru', LOCAL_RES + '/.', DRIVE_RES + '/.'],
                   check=True, capture_output=True)
    if not silent:
        print('OK')


_sync_stop  = threading.Event()
_sync_thread = None

def _background_sync_loop(interval_min=10):
    while not _sync_stop.wait(timeout=interval_min * 60):
        ts = time.strftime('%H:%M:%S')
        try:
            sync_to_drive(silent=True)
            print(f'  [sync auto {ts}] Drive actualizado', flush=True)
        except Exception as e:
            print(f'  [sync auto {ts}] ERRO: {e}', flush=True)

def start_background_sync(interval_min=10):
    global _sync_thread, _sync_stop
    _sync_stop = threading.Event()
    _sync_thread = threading.Thread(
        target=_background_sync_loop,
        args=(interval_min,),
        daemon=True
    )
    _sync_thread.start()
    print(f'Sync automático iniciado (a cada {interval_min} min).')

def stop_background_sync():
    global _sync_thread
    if _sync_thread and _sync_thread.is_alive():
        _sync_stop.set()
        _sync_thread.join(timeout=30)
    sync_to_drive()
    print('Sync automático parado. Drive actualizado.')



def ensure_root_files():
    drive_ds = os.path.join(DRIVE_PRE, DS_NAME)
    for fname in ['dataset.json', 'dataset_fingerprint.json', 'splits_final.json',
                  'nnUNetPlans.json', 'nnUNetResEncUNetMPlans.json', 'nnUNetResEncUNetLPlans.json']:
        src = os.path.join(drive_ds, fname)
        dst = os.path.join(DS_PRE, fname)
        if os.path.isfile(src) and not os.path.isfile(dst):
            shutil.copy2(src, dst)
            print(f'  copiado: {fname}')


def ensure_preprocessed(config_folder):
    """Copia subpasta de preprocessado do Drive → /content/ se necessário.

    Retorna True se disponível, False se não encontrado no Drive.
    """
    dst = os.path.join(DS_PRE, config_folder)
    if os.path.isdir(dst) and os.listdir(dst):
        return True
    src = os.path.join(DRIVE_PRE, DS_NAME, config_folder)
    if not os.path.isdir(src):
        print(f'  [AVISO] {config_folder} não encontrado no Drive.')
        print(f'          Use a Célula PREPROC para gerar no Colab.')
        return False
    print(f'  Copiando {config_folder} do Drive (pode demorar)...')
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['cp', '-ru', src + '/.', dst + '/.'], check=True)
    return True


def is_done(trainer, plans_id, config, fold):
    folder = f'{trainer}__{plans_id}__{config}'
    return os.path.isfile(os.path.join(DS_RES, folder, f'fold_{fold}', 'checkpoint_final.pth'))


def has_checkpoint(trainer, plans_id, config, fold):
    folder = f'{trainer}__{plans_id}__{config}'
    return os.path.isfile(os.path.join(DS_RES, folder, f'fold_{fold}', 'checkpoint_latest.pth'))


def train_fold(config, fold, plans_id='nnUNetPlans', trainer='nnUNetTrainer', label='?'):
    """Treina um fold. Salta se já concluído; retoma se tiver checkpoint_latest.

    Sincroniza para Drive após conclusão (o sync em background já protege durante o treino).
    """
    if is_done(trainer, plans_id, config, fold):
        print(f'  [{label}] fold {fold}: JÁ CONCLUÍDO ✓')
        return True

    estado = 'retomando de checkpoint' if has_checkpoint(trainer, plans_id, config, fold) else 'início do zero'
    print(f'  [{label}] fold {fold}: {estado}...')

    cmd = ['nnUNetv2_train', str(DS_ID), config, str(fold), '--npz']
    if plans_id != 'nnUNetPlans':
        cmd += ['-p', plans_id]
    if trainer != 'nnUNetTrainer':
        cmd += ['-tr', trainer]

    result = subprocess.run(cmd)
    sync_to_drive()

    if result.returncode == 0:
        print(f'  [{label}] fold {fold}: CONCLUÍDO ✓')
        return True
    else:
        print(f'  [{label}] fold {fold}: FALHOU (código {result.returncode})')
        return False


def print_status():
    ARCHS = [
        ('3d_lowres',          'nnUNetPlans',            'nnUNetTrainer_Foreground100', 'lowres CF-prereq'),
        ('3d_lowres',          'nnUNetResEncUNetMPlans',  'nnUNetTrainer_Foreground100', 'lowres CFREM-prereq'),
        ('2d',                 'nnUNetPlans',             'nnUNetTrainer',               '2D'),
        ('2d',                 'nnUNetResEncUNetMPlans',  'nnUNetTrainer',               '2D_REM'),
        ('2d',                 'nnUNetResEncUNetLPlans',  'nnUNetTrainer',               '2D_REL'),
        ('3d_fullres',         'nnUNetPlans',             'nnUNetTrainer',               '3D_F'),
        ('3d_fullres',         'nnUNetResEncUNetMPlans',  'nnUNetTrainer',               '3D_FREM'),
        ('3d_cascade_fullres', 'nnUNetPlans',             'nnUNetTrainer_Foreground100', '3D_CF'),
        ('3d_cascade_fullres', 'nnUNetResEncUNetMPlans',  'nnUNetTrainer_Foreground100', '3D_CFREM'),
    ]
    print('\n=== ESTADO (█=concluído  ~=retomável  ░=pendente) ===')
    total = 0
    for config, plans, trainer, label in ARCHS:
        bar = ''.join(
            '█' if is_done(trainer, plans, config, f) else
            '~' if has_checkpoint(trainer, plans, config, f) else '░'
            for f in range(5)
        )
        n = bar.count('█')
        total += n
        print(f'  {label:20s} [{bar}] {n}/5')
    print(f'  Total: {total}/45 folds')
    print()


print('Célula C OK')

---
## Célula D — Preparar ambiente
**Execute sempre ao iniciar sessão.**

Copia checkpoints e ficheiros JSON de configuração do Drive para `/content/`.

In [ ]:
print('=== Preparando ambiente ===')

if os.path.isdir(DRIVE_RES):
    print('Copiando checkpoints do Drive (sem checkpoint_best)...')
    os.makedirs(LOCAL_RES, exist_ok=True)
    bash_cmd = (
        f'find "{DRIVE_RES}" -type f ! -name "checkpoint_best.pth" | while IFS= read -r src; do '
        f'  dst="{LOCAL_RES}/${{src#{DRIVE_RES}/}}"; '
        f'  mkdir -p "$(dirname "$dst")"; '
        f'  cp -n "$src" "$dst"; '
        f'done'
    )
    subprocess.run(['bash', '-c', bash_cmd], check=True)
    print('OK')
else:
    print('Nenhum resultado no Drive — início do zero.')

print('Copiando ficheiros JSON de configuração...')
ensure_root_files()

print_status()

---
## TREINAR TUDO

Corre a pipeline completa — do início ao fim.

- O que **já está concluído** (`checkpoint_final.pth`) é **saltado**
- O que tem **checkpoint parcial** (`checkpoint_latest.pth`) é **retomado** de onde parou
- O que não existe ainda começa **do zero**

Pode re-executar esta célula a qualquer momento (nova sessão, interrupção) — retoma sempre de onde parou.

In [ ]:
print('=== PIPELINE COMPLETA ===')
print()

start_background_sync(interval_min=10)

try:

    print('--- FASE 1: 3d_lowres nnUNetPlans + Foreground100 ---')
    if ensure_preprocessed('nnUNetPlans_3d_lowres'):
        for fold in range(5):
            train_fold('3d_lowres', fold,
                       plans_id='nnUNetPlans',
                       trainer='nnUNetTrainer_Foreground100',
                       label='lowres-CF')
    print()

    print('--- FASE 2: 3d_lowres nnUNetResEncUNetMPlans + Foreground100 ---')
    if ensure_preprocessed('nnUNetResEncUNetMPlans_3d_lowres'):
        for fold in range(5):
            train_fold('3d_lowres', fold,
                       plans_id='nnUNetResEncUNetMPlans',
                       trainer='nnUNetTrainer_Foreground100',
                       label='lowres-CFREM')
    print()

    print('--- FASE 3: 2D nnUNetPlans ---')
    if ensure_preprocessed('nnUNetPlans_2d'):
        for fold in range(5):
            train_fold('2d', fold, plans_id='nnUNetPlans', trainer='nnUNetTrainer', label='2D')
    print()

    print('--- FASE 4: 2D_REM nnUNetResEncUNetMPlans ---')
    if ensure_preprocessed('nnUNetResEncUNetMPlans_2d'):
        for fold in range(5):
            train_fold('2d', fold, plans_id='nnUNetResEncUNetMPlans', trainer='nnUNetTrainer', label='2D_REM')
    print()

    print('--- FASE 5: 2D_REL nnUNetResEncUNetLPlans ---')
    if ensure_preprocessed('nnUNetResEncUNetLPlans_2d'):
        for fold in range(5):
            train_fold('2d', fold, plans_id='nnUNetResEncUNetLPlans', trainer='nnUNetTrainer', label='2D_REL')
    print()

    print('--- FASE 6: 3D_F nnUNetPlans 3d_fullres ---')
    if ensure_preprocessed('nnUNetPlans_3d_fullres'):
        for fold in range(5):
            train_fold('3d_fullres', fold, plans_id='nnUNetPlans', trainer='nnUNetTrainer', label='3D_F')
    print()

    print('--- FASE 7: 3D_FREM nnUNetResEncUNetMPlans 3d_fullres ---')
    if ensure_preprocessed('nnUNetResEncUNetMPlans_3d_fullres'):
        for fold in range(5):
            train_fold('3d_fullres', fold, plans_id='nnUNetResEncUNetMPlans', trainer='nnUNetTrainer', label='3D_FREM')
    print()

    print('--- FASE 8: 3D_CF cascade nnUNetPlans + Foreground100 ---')
    if not all(is_done('nnUNetTrainer_Foreground100', 'nnUNetPlans', '3d_lowres', f) for f in range(5)):
        print('  AGUARDANDO Fase 1 (5/5). Re-execute esta célula quando concluída.')
    elif ensure_preprocessed('nnUNetPlans_3d_fullres'):
        for fold in range(5):
            train_fold('3d_cascade_fullres', fold,
                       plans_id='nnUNetPlans',
                       trainer='nnUNetTrainer_Foreground100',
                       label='3D_CF')
    print()

    print('--- FASE 9: 3D_CFREM cascade nnUNetResEncUNetMPlans + Foreground100 ---')
    if not all(is_done('nnUNetTrainer_Foreground100', 'nnUNetResEncUNetMPlans', '3d_lowres', f) for f in range(5)):
        print('  AGUARDANDO Fase 2 (5/5). Re-execute esta célula quando concluída.')
    elif ensure_preprocessed('nnUNetResEncUNetMPlans_3d_fullres'):
        for fold in range(5):
            train_fold('3d_cascade_fullres', fold,
                       plans_id='nnUNetResEncUNetMPlans',
                       trainer='nnUNetTrainer_Foreground100',
                       label='3D_CFREM')

finally:
    stop_background_sync()

print()
print_status()

---
## Célula PREPROC — Pré-processar configs em falta

**Só executar se a célula TREINAR TUDO avisar que um preprocessado está em falta.**

O RunPod estava a gerar no Drive:
- `nnUNetResEncUNetMPlans_2d` (Fase 4 / 2D_REM)
- `nnUNetResEncUNetLPlans_2d` (Fase 5 / 2D_REL)
- `nnUNetResEncUNetMPlans_3d_fullres` (Fase 7 / 3D_FREM)

Se não estiverem disponíveis, esta célula executa o pré-processamento no Colab
usando os dados LIDC em `Drive/Pulmonary/data/processed/lidc_idri/`.
Após conclusão, voltar a executar **TREINAR TUDO**.

In [ ]:
import json

DRIVE_LIDC   = os.path.join(DRIVE, 'data', 'processed', 'lidc_idri')
LOCAL_LIDC   = '/content/lidc_idri'
LOCAL_RAW_DS = os.path.join(LOCAL_RAW, DS_NAME)

if not os.path.isdir(DRIVE_LIDC) or not os.listdir(DRIVE_LIDC):
    print('AVISO: dados LIDC não encontrados em Drive/Pulmonary/data/processed/lidc_idri/')
    print('O RunPod pode ainda estar a fazer o upload. Aguarde e re-execute.')
else:
    if not os.path.isdir(LOCAL_LIDC) or not os.listdir(LOCAL_LIDC):
        print('Copiando dados LIDC do Drive (~45 GB, ~15-20 min)...')
        os.makedirs(LOCAL_LIDC, exist_ok=True)
        subprocess.run(['cp', '-ru', DRIVE_LIDC + '/.', LOCAL_LIDC + '/.'], check=True)
        print('LIDC copiado.')
    else:
        print('LIDC já em /content/.')

    os.makedirs(LOCAL_RAW_DS + '/imagesTr', exist_ok=True)
    os.makedirs(LOCAL_RAW_DS + '/labelsTr', exist_ok=True)
    dj_src = os.path.join(DS_PRE, 'dataset.json')
    dj_dst = os.path.join(LOCAL_RAW_DS, 'dataset.json')
    if os.path.isfile(dj_src) and not os.path.isfile(dj_dst):
        shutil.copy2(dj_src, dj_dst)

    with open(os.path.join(DS_PRE, 'splits_final.json')) as f:
        splits = json.load(f)
    dev_ids = set()
    for fold_data in splits:
        dev_ids.update(fold_data['train'])
        dev_ids.update(fold_data['val'])

    import SimpleITK as sitk
    n = 0
    for pid in sorted(dev_ids):
        ct_src  = os.path.join(LOCAL_LIDC, 'images', pid, 'ct.nii.gz')
        lbl_src = os.path.join(LOCAL_LIDC, 'labels', pid, 'consensus.nii.gz')
        ct_dst  = os.path.join(LOCAL_RAW_DS, 'imagesTr', f'{pid}_0000.nii.gz')
        lbl_dst = os.path.join(LOCAL_RAW_DS, 'labelsTr', f'{pid}.nii.gz')
        if not os.path.isfile(ct_src):
            continue
        if not os.path.exists(ct_dst):
            os.symlink(ct_src, ct_dst)
        if os.path.isfile(lbl_src) and not os.path.exists(lbl_dst):
            os.symlink(lbl_src, lbl_dst)
        elif not os.path.exists(lbl_dst):
            ct_img = sitk.ReadImage(ct_src)
            empty  = sitk.Image(ct_img.GetSize(), sitk.sitkUInt8)
            empty.CopyInformation(ct_img)
            sitk.WriteImage(empty, lbl_dst, True)
        n += 1
    print(f'{n} casos ligados em imagesTr/labelsTr.')

    MISSING = [
        ('2d',         'nnUNetPlannerResEncM', 'nnUNetResEncUNetMPlans_2d',         '2D_REM'),
        ('2d',         'nnUNetPlannerResEncL', 'nnUNetResEncUNetLPlans_2d',         '2D_REL'),
        ('3d_fullres', 'nnUNetPlannerResEncM', 'nnUNetResEncUNetMPlans_3d_fullres', '3D_FREM'),
    ]

    for cfg, planner, folder, label in MISSING:
        dst = os.path.join(DS_PRE, folder)
        if os.path.isdir(dst) and os.listdir(dst):
            print(f'  [{label}] já disponível — pulando')
            continue
        print(f'  [{label}] pré-processando ({cfg} / {planner})...')
        r = subprocess.run(
            ['nnUNetv2_plan_and_preprocess', '-d', str(DS_ID),
             '-c', cfg, '-pl', planner, '--verify_dataset_integrity'],
            capture_output=True, text=True
        )
        if r.returncode == 0:
            print(f'  [{label}] OK — copiando para Drive...')
            drive_dst = os.path.join(DRIVE_PRE, DS_NAME, folder)
            os.makedirs(drive_dst, exist_ok=True)
            subprocess.run(['cp', '-ru', dst + '/.', drive_dst + '/.'])
            print(f'  [{label}] Drive actualizado.')
        else:
            print(f'  [{label}] ERRO:\n{r.stderr[-500:]}')

    print('\nPré-processamento concluído. Voltar a executar a célula TREINAR TUDO.')

---
## Célula STATUS — Progresso e sync manual

In [ ]:
print_status()
sync_to_drive()
print('Drive actualizado.')